# 1.10 參數的影響怎樣穿過兩段計算？


接字表的分數要先變成機率，再變成代價；參數與代價中間可能隔很多運算。先用兩段透明的材料看影響怎麼傳：`u=2*w`，接著 `L=u²`。當 w=3，前向得到 u=6、L=36。

如果 w 微增 0.001，第一段讓 u 增 0.002。第二段在 u=6 附近，每增加一單位 u，L 約增加 `2*u=12` 單位；所以 L 約增 `12×0.002=0.024`。相對原來的 w 小步，總敏感度為 `0.024/0.001≈24`。

每段的敏感度相乘，就是**鏈式法則**：`dL/dw=(dL/du)×(du/dw)=12×2=24`。符號 `dL/du` 是「對 u 的每單位變化，L 如何變」，不是把 L 除以 u。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
# @title 本節圖解
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/rewrite-01-chain.svg")))

圖上方的 3→6→36 是數值；下方的 12×2=24 才是敏感度。兩者不能混讀。下面用 PyTorch 自動核對這條運算關係。


In [ ]:
import torch

w = torch.tensor(3.0, requires_grad=True)
u = 2 * w
loss = u.square()
loss.backward()
print("中間值", u.item(), "代價", loss.item())
print("兩段敏感度", 2 * u.detach().item(), 2)
print("合成梯度", w.grad.item())
assert w.grad.item() == 24

`requires_grad=True` 要求儲存 w 參與的運算關係。先計算 u 與 L 的方向叫**前向計算**；`loss.backward()` 沿關係反向求導，把 24 存到 `w.grad`。輸出依序是中間值 6、代價 36，兩段敏感度 12 與 2，總梯度 24。

PyTorch 這套工具叫 **autograd**，儲存的運算連線叫**計算圖**。它用每個運算已知的求導規則往回算，不是靠左右試參數。`.item()` 取出單個普通數字；`.detach()` 只在顯示時停止追蹤這份值的求導關係，不會改 w。

現在 w 仍是 3：`backward()` 算出影響量，沒有把參數搬到新位置。

練習將 `u=2*w` 改成 `u=3*w`，顯示的第一段敏感度 2 也改成 3。w=3 時，u=9、L=81，梯度應為 `18×3=54`，把斷言同步改成 54 再核對。不能只把舊梯度乘 1.5，因為第二段所在的 u 也變了。

<details>
<summary>補充與重做</summary>

本例只有一條路。若同一參數沿兩條路影響同一結果，各路用鏈式法則求出影響後還要相加；[4.2](https://birdhackor.github.io/tiny-perceptron-vlm/4.2.html) 會看一個例子。這裡的平方代價是手算材料，文字模型則仍使用 [1.8](https://birdhackor.github.io/tiny-perceptron-vlm/1.8.html) 的答案代價。

</details>
